<a href="https://colab.research.google.com/github/bendragstrem-ux/AI-HW-BKD/blob/main/Benjamin_Dragstrem_HW5_8515_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
https://api.fda.gov/drug/event.json?search=patient.drug.openfda.generic_name:"naloxone"&limit=100&skip=0

In [1]:
import requests
import time
import json

def fetch_url_with_retry(url, retries=3, backoff_factor=0.5):
    """
    Fetches content from a URL with retry logic.

    Args:
        url (str): The URL to fetch.
        retries (int): Number of retries on failure.
        backoff_factor (float): Factor for exponential backoff between retries.

    Returns:
        dict: Parsed JSON response if successful.
        None: If all retries fail.
    """
    for i in range(retries + 1):
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt {i + 1}/{retries + 1} failed: {e}")
            if i < retries:
                sleep_time = backoff_factor * (2 ** i)
                print(f"Retrying in {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
            else:
                print("All retries failed.")
                return None

print("Defined fetch_url_with_retry function.")

Defined fetch_url_with_retry function.


In [4]:
# Example usage of the fetch_url_with_retry function
api_url = '''https://api.fda.gov/drug/event.json?search=patient.drug.openfda.generic_name:"naloxone"&limit=100&skip=0'''

data = fetch_url_with_retry(api_url)

if data:
    print("Successfully fetched data. First 500 characters:")
    print(json.dumps(data, indent=2)[:500])
else:
    print("Failed to fetch data after multiple retries.")

Successfully fetched data. First 500 characters:
{
  "meta": {
    "disclaimer": "Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.",
    "terms": "https://open.fda.gov/terms/",
    "license": "https://open.fda.gov/license/",
    "last_updated": "2026-04-28",
    "results": {
      "skip": 0,
      "limit": 100,
      "total": 35830


In [5]:
import math

def fetch_all_paginated_data(base_url, limit=100):
    all_results = []
    skip = 0
    total_records = None

    print(f"Starting data fetch from {base_url.split('?')[0]}...")

    while True:
        paginated_url = f"{base_url.split('&skip=')[0]}&skip={skip}"
        print(f"Fetching page (skip={skip}, limit={limit})...")
        data = fetch_url_with_retry(paginated_url)

        if data is None:
            print("Failed to fetch data for current page. Stopping pagination.")
            break

        if total_records is None and 'meta' in data and 'results' in data['meta']:
            total_records = data['meta']['results'].get('total')
            print(f"Total records available: {total_records}")

        if 'results' in data and data['results']:
            all_results.extend(data['results'])
            print(f"Fetched {len(data['results'])} records. Total collected: {len(all_results)}.")
        else:
            print("No more results found or 'results' key missing.")
            break

        skip += limit

        if total_records is not None and skip >= total_records:
            print("Reached or exceeded total records. Stopping pagination.")
            break

        # openFDA has a max skip of 25000, so we should also stop there if needed
        if skip > 25000:
            print("Reached maximum skip limit (25,000) for openFDA API. Stopping pagination.")
            break

    print(f"Finished fetching data. Collected {len(all_results)} records.")
    return all_results

# Call the pagination function with the previously defined API URL
base_api_url = 'https://api.fda.gov/drug/event.json?search=patient.drug.openfda.generic_name:"naloxone"&limit=100&skip=0'
all_naloxone_events = fetch_all_paginated_data(base_api_url, limit=100)

# Save the raw JSON responses to a file
output_json_filepath = '/content/openfda_naloxone_events.json'
with open(output_json_filepath, 'w') as f:
    json.dump(all_naloxone_events, f, indent=2)

print(f"All fetched data saved to {output_json_filepath}.")
print(f"Total records saved: {len(all_naloxone_events)}")

Starting data fetch from https://api.fda.gov/drug/event.json...
Fetching page (skip=0, limit=100)...
Total records available: 35830
Fetched 100 records. Total collected: 100.
Fetching page (skip=100, limit=100)...
Fetched 100 records. Total collected: 200.
Fetching page (skip=200, limit=100)...
Fetched 100 records. Total collected: 300.
Fetching page (skip=300, limit=100)...
Fetched 100 records. Total collected: 400.
Fetching page (skip=400, limit=100)...
Fetched 100 records. Total collected: 500.
Fetching page (skip=500, limit=100)...
Fetched 100 records. Total collected: 600.
Fetching page (skip=600, limit=100)...
Fetched 100 records. Total collected: 700.
Fetching page (skip=700, limit=100)...
Fetched 100 records. Total collected: 800.
Fetching page (skip=800, limit=100)...
Fetched 100 records. Total collected: 900.
Fetching page (skip=900, limit=100)...
Fetched 100 records. Total collected: 1000.
Fetching page (skip=1000, limit=100)...
Fetched 100 records. Total collected: 1100.
Fe

In [6]:
import pandas as pd

# Load the JSON data that was just saved
with open('/content/openfda_naloxone_events.json', 'r') as f:
    naloxone_events_json = json.load(f)

print(f"Loaded {len(naloxone_events_json)} records from JSON for conversion to CSV.")

# Flatten the JSON data into a pandas DataFrame
# The openFDA API results are often nested, so we'll attempt to flatten them.
# For a full flattening, more complex logic might be needed, but pd.json_normalize is a good start.
# We will normalize up to one level for simplicity to create flat rows.

# Let's inspect a sample record to understand its structure if needed
# print(json.dumps(naloxone_events_json[0], indent=2))

# Use json_normalize to flatten the data
df = pd.json_normalize(naloxone_events_json)

# Save the DataFrame to a CSV file
output_csv_filepath = '/content/openfda_naloxone_events.csv'
df.to_csv(output_csv_filepath, index=False)

print(f"Data successfully converted to CSV and saved to {output_csv_filepath}.")

# Load the CSV with pandas and print the record count
df_from_csv = pd.read_csv(output_csv_filepath)
print(f"Loaded CSV has {len(df_from_csv)} records.")

# Display the first few rows and columns to confirm structure
print("\nFirst 5 rows of the CSV data:")
print(df_from_csv.head())
print("\nColumns in the CSV data:")
print(df_from_csv.columns.tolist())

Loaded 25100 records from JSON for conversion to CSV.
Data successfully converted to CSV and saved to /content/openfda_naloxone_events.csv.
Loaded CSV has 25100 records.

First 5 rows of the CSV data:
   safetyreportversion  safetyreportid primarysourcecountry occurcountry  \
0                    3        10004454                   IT           IT   
1                    4        10004741                   US           US   
2                    1        10005426                   US           US   
3                    3        10007523                   NO           NO   
4                    2        10010895                   US          NaN   

   transmissiondateformat  transmissiondate  reporttype  serious  \
0                     102          20150326           1        1   
1                     102          20141212           2        1   
2                     102          20141002           1        1   
3                     102          20141212           2        1   
4 

## Roadmap Step 7: Markdown Answers

### 1. Pagination Strategy and Stop Condition

Our pagination strategy for the openFDA API involved iteratively fetching data by incrementing the `skip` parameter in the URL. We set a `limit` of 100 records per request. The loop continued as long as new data was returned and our `skip` value had not yet reached the `total` number of records reported by the API in the `meta.results.total` field.

The primary stopping condition was when the `skip` value became greater than or equal to the `total` records reported by the API. Additionally, openFDA has a hard limit of 25,000 for the `skip` parameter. Therefore, an explicit check was included to stop pagination if `skip` exceeded 25,000, ensuring we do not make invalid requests and handle API-specific constraints gracefully. In this particular execution, the `skip` limit of 25,000 was reached, and `25,100` records were collected before the `total` records of `35,830` could be fully retrieved.

### 2. Retry Logic and its Importance in Production

The `fetch_url_with_retry` function was implemented to make robust API requests. It retries failed requests up to 3 times, with an exponential backoff (starting at 0.5 seconds, then 1, then 2). This retry mechanism is crucial in a production environment for several reasons:

*   **Network Instability:** Temporary network glitches, DNS resolution issues, or server hiccups are common. Retries prevent immediate failure and allow transient issues to resolve themselves.
*   **API Rate Limits:** While we didn't explicitly implement rate limiting, retries can help manage momentary spikes in request volume that might lead to `429 Too Many Requests` errors from the API.
*   **Server Load:** APIs can experience periods of high load, leading to slower responses or temporary errors. Retries with backoff give the server time to recover.
*   **Increased Reliability:** It significantly increases the reliability and resilience of data ingestion pipelines, reducing the need for manual intervention and ensuring more complete data collection.
*   **Graceful Degradation:** Instead of crashing, the application can gracefully handle temporary external service interruptions.

### 3. Status Codes Actually Seen

During the execution, the `requests.raise_for_status()` method was used, which would raise an `HTTPError` for 4xx or 5xx client and server error responses. Since the pagination completed successfully up to the API's skip limit, this indicates that all API requests returned a `200 OK` status code, meaning the requests were successful. No error status codes were encountered that triggered the retry logic or caused a hard stop.

### 4. ETL/ELT Positioning

This entire process—fetching data from the openFDA API, saving it as raw JSON, transforming it into a flat CSV, and loading it into a pandas DataFrame—fits squarely into the **Extract, Transform, Load (ETL)** paradigm, or more specifically, **Extract, Load, Transform (ELT)**.

*   **Extract:** The `fetch_url_with_retry` and `fetch_all_paginated_data` functions handle the extraction of raw data from the openFDA API.
*   **Load (to JSON):** Saving the raw API responses to `/content/openfda_naloxone_events.json` represents an initial 'Load' phase, storing the data in its original, raw format.
*   **Transform:** The use of `pd.json_normalize` to flatten the nested JSON structure into a tabular format, and then saving it as `/content/openfda_naloxone_events.csv`, represents the 'Transform' step. This process prepares the data for easier analysis and storage in a structured format like a database or data warehouse.
*   **Load (to CSV/DataFrame):** Finally, reading the CSV back into a pandas DataFrame (`df_from_csv`) is the ultimate 'Load' into an analytical environment.